# 📚 LangChain Document Loading & Chunking — Complete RAG Practical Notebook

This notebook is designed as a **practical teaching/demo notebook for RAG**.

We will cover two major stages:

```text
                RAG DATA PREPARATION
                        │
             ┌──────────┴──────────┐
             ▼                     ▼
       DOCUMENT LOADING        TEXT CHUNKING
             │                     │
     ┌───────┼────────┐      ┌─────┼──────────────┐
     ▼       ▼        ▼      ▼     ▼              ▼
   TXT      PDF      DOCX   Character Recursive   Token
                              │       │              │
                              └───────┴──────────────┘
                                      ▼
                                  Embeddings
                                      ▼
                                 Vector Store
                                      ▼
                                     RAG
```

## Contents

### Part 1 — Document Loaders
1. Text Loader
2. PDF Loader
3. DOCX Loader
4. Web Loader

### Part 2 — Text Splitters
5. CharacterTextSplitter
6. RecursiveCharacterTextSplitter
7. Token-based splitting
8. Character vs Recursive vs Token comparison
9. Choosing the right splitter for RAG

> **Important:** Run the setup/import cells first. Loader examples that depend on external files or a live website require the corresponding file/package/network access.


# 1️⃣ Install Required Packages

Run this if these packages are not already installed.

```bash
pip install -U langchain langchain-community langchain-text-splitters
pip install pypdf docx2txt beautifulsoup4
pip install tiktoken
```

### Package purpose

| Package | Purpose |
|---|---|
| `langchain` | Core LangChain functionality |
| `langchain-community` | Community document loaders |
| `langchain-text-splitters` | Text splitting implementations |
| `pypdf` | PDF parsing |
| `docx2txt` | DOCX text extraction |
| `beautifulsoup4` | HTML parsing |
| `tiktoken` | Token counting for supported tokenizers |

If you use a non-OpenAI model, tokenization may differ. Token limits should be checked using the tokenizer/model-specific method where available.


In [2]:
# Core imports

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    WebBaseLoader,
)

print("Imports successful!")


C:\Users\HP\AppData\Local\Temp\ipykernel_16760\3079081025.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


Imports successful!


# 2️⃣ The RAG Data Preparation Pipeline

Before learning individual loaders and splitters, understand the complete flow:

```text
                   SOURCE DATA
                       │
       ┌───────────────┼────────────────┐
       │               │                │
       ▼               ▼                ▼
      TXT             PDF              DOCX
       │               │                │
       └───────────────┼────────────────┘
                       │
                       ▼
                 DOCUMENT LOADER
                       │
                       ▼
               LangChain Document
             ┌────────────────────┐
             │ page_content       │
             │ metadata           │
             └────────────────────┘
                       │
                       ▼
                  TEXT SPLITTER
                       │
           ┌───────────┼───────────┐
           ▼           ▼           ▼
       Character   Recursive     Token
           │           │           │
           └───────────┼───────────┘
                       ▼
                    CHUNKS
                       │
                       ▼
                   EMBEDDINGS
                       │
                       ▼
                 VECTOR DATABASE
                       │
                       ▼
                      RAG
```


# 3️⃣ Understanding the LangChain `Document`

Most LangChain document loaders return a list of `Document` objects.

Conceptually:

```text
Document
│
├── page_content
│      └── Actual extracted text
│
└── metadata
       ├── source
       ├── page
       └── other source information
```

Example:

```python
documents = loader.load()

print(type(documents))
print(type(documents[0]))
print(documents[0].page_content)
print(documents[0].metadata)
```

This distinction becomes important when we pass the extracted content into a splitter.


# 4️⃣ TXT — Loading a Text File

For a plain `.txt` file:

```python
TextLoader("data/data.txt")
```

Flow:

```text
sample_data.txt
      │
      ▼
  TextLoader
      │
      ▼
 list[Document]
      │
      ├── page_content
      └── metadata
```

### Code


In [15]:
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader("data/sample_data.txt")

text_docs = text_loader.load()

print(type(text_docs))
print("Documents:", len(text_docs))
print("Characters:", len(text_docs[0].page_content))
print("Metadata:", text_docs[0].metadata)

print("\n--- CONTENT ---")
print(text_docs[0].page_content)


<class 'list'>
Documents: 1
Characters: 851
Metadata: {'source': 'data/sample_data.txt'}

--- CONTENT ---
ACME Technologies â€” Employee Policy

1. Working Hours

Employees are expected to work from 9:00 AM to 6:00 PM,
Monday through Friday. Employees should be available during
core working hours from 10:00 AM to 4:00 PM.

2. Work From Home

Employees can work remotely for up to three days per week.
Remote work must be approved by the employee's manager.

3. Leave Policy

Employees receive 24 days of paid leave per calendar year.
Leave requests should normally be submitted at least three
working days in advance.

4. Security Policy

Employees must use company-managed devices when accessing
confidential company information. Passwords must not be
shared with other employees.

5. Learning and Development

Employees receive an annual learning budget of INR 25,000.
The budget can be used for approved courses, certifications,
books, and conferences.


# 5️⃣ PDF — Loading a PDF

For PDFs, a common loader is:

```python
PyPDFLoader("data/data.pdf")
```

A useful difference from a TXT file:

```text
PDF
 │
 ├── Page 1 ──► Document
 ├── Page 2 ──► Document
 ├── Page 3 ──► Document
 └── ...
```

So a PDF loader commonly gives you one `Document` per page.

That page-level structure is useful because metadata can preserve the page number.


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("data/data.pdf")

pdf_docs = pdf_loader.load()

print("Number of PDF Documents:", len(pdf_docs))

for i, doc in enumerate(pdf_docs[:3], start=1):
    print(f"\n--- PDF Document {i} ---")
    print("Characters:", len(doc.page_content))
    print("Metadata:", doc.metadata)
    print("Preview:", doc.page_content[:300])


Number of PDF Documents: 2

--- PDF Document 1 ---
Characters: 946
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-18T22:07:04+05:30', 'author': 'HP', 'moddate': '2026-09-18T22:07:04+05:30', 'source': 'data/data.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
Preview: ABC Technologies – Employee Policy Document 
 
1. Work From Home Policy 
 
Employees can work from home up to 2 days per week with prior approval from their manager. 
 
Employees must be available during the core working hours of 10:00 AM to 6:00 PM. 
 
Any additional work-from-home day requires app

--- PDF Document 2 ---
Characters: 1175
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-18T22:07:04+05:30', 'author': 'HP', 'moddate': '2026-09-18T22:07:04+05:30', 'source': 'data/data.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}
Preview: Employees receive a performance review twice a year. 
 
T

## PDF Loader Flow

```text
              sample.pdf
                  │
                  ▼
             PyPDFLoader
                  │
       ┌──────────┼──────────┐
       ▼          ▼          ▼
     Page 1     Page 2     Page 3
       │          │          │
       ▼          ▼          ▼
    Document   Document   Document
       │          │          │
       └──────────┼──────────┘
                  ▼
            Text Splitter
```


# 6️⃣ DOCX — Loading a Word Document

For `.docx`:

```python
Docx2txtLoader("data/data.docx")
```

Typical flow:

```text
company_policy.docx
        │
        ▼
 Docx2txtLoader
        │
        ▼
    Document
        │
        ├── page_content
        └── metadata
```

Unlike a PDF, the extracted structure depends on the loader and source document. Do not assume DOCX "pages" correspond exactly to Word's visual pages after extraction.


In [7]:
from langchain_community.document_loaders import Docx2txtLoader

docx_loader = Docx2txtLoader("data/data.docx")
docs = docx_loader.load()

print("Number of DOCX Documents:", len(docs))

print("Characters:", len(doc.page_content))
print("Metadata:", doc.metadata)
print("Preview:", doc.page_content[:500])


Number of DOCX Documents: 1
Characters: 2122
Metadata: {'source': 'data/data.docx'}
Preview: ABC Technologies – Employee Policy Document



1. Work From Home Policy



Employees can work from home up to 2 days per week with prior approval from their manager.



Employees must be available during the core working hours of 10:00 AM to 6:00 PM.



Any additional work-from-home day requires approval from the department head.



2. Leave Policy



Employees receive 18 paid leaves per year.



Employees should apply for planned leave at least 3 working days in advance.



For emergency leave,


# 7️⃣ WEB — Loading a Web Page

LangChain can also load web content.

Example:

```python
loader = WebBaseLoader("https://example.com")
docs = loader.load()
```

Conceptually:

```text
             URL
              │
              ▼
       WebBaseLoader
              │
              ▼
       HTML / webpage
              │
              ▼
       Extracted text
              │
              ▼
          Document
              │
              ▼
        Text Splitter
```

### Important

Web pages contain navigation, menus, footers, advertisements, scripts, and other content. In production RAG systems, web extraction/cleaning may require additional processing depending on the site.


In [8]:
from langchain_community.document_loaders import WebBaseLoader

# Example public webpage.
# Replace with a page you are allowed to access.

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"

web_loader = WebBaseLoader(url)

web_docs = web_loader.load()

print("Number of web documents:", len(web_docs))
print("Metadata:", web_docs[0].metadata)
print("Preview:")
print(web_docs[0].page_content[:1000])


Number of web documents: 1
Metadata: {'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}
Preview:




Artificial intelligence - Wikipedia


























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in





























Contents
move to sidebar
hide




(Top)





1
Goals




Toggle Goals subsection





1.1
Reasoning and problem-solving








1.2
Knowledge representation








1.3
Planning and decision-making








1.4
Learning








1.5
Natural language processing








1.6
Perception





# 8️⃣ Compare the Four Loaders

| Source | Loader | Typical output |
|---|---|---|
| TXT | `TextLoader` | Document containing text |
| PDF | `PyPDFLoader` | Commonly one Document per page |
| DOCX | `Docx2txtLoader` | Extracted document text |
| Web | `WebBaseLoader` | Extracted webpage content |


# 9️⃣ Why Do We Need Chunking?

Suppose a document contains:

```text
50,000 characters
```

Sending the entire document to an LLM for every query can cause:

- large prompts
- higher cost
- higher latency
- context-window pressure
- irrelevant information
- weaker retrieval precision

Instead:

```text
50,000-character document
          │
          ▼
      Splitter
          │
     ┌────┼────┐
     ▼    ▼    ▼
   Chunk Chunk Chunk ...
     │    │    │
     └────┼────┘
          ▼
      Embeddings
          ▼
     Vector Store
```

At query time, we retrieve only the relevant chunks.


# 🔟 CharacterTextSplitter

## Basic code

```python
splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(text)
```

### Critical concept

`chunk_size` should be understood as a **target used during splitting/merging**, not as a universal guarantee that every final chunk will never exceed that number.

The separator is extremely important.


## CharacterTextSplitter Flow

With:

```python
separator="\n\n"
chunk_size=50
chunk_overlap=0
```

think of the process as:

```text
FULL TEXT
   │
   ▼
Find "\n\n"
   │
   ▼
Split into separator-defined pieces
   │
   ├────────────┐
   ▼            ▼
Paragraph 1  Paragraph 2
   │            │
   └──────┬─────┘
          ▼
    Merge pieces
    toward target
    chunk_size=50
          │
          ▼
     Final chunks
```

### Why can you get 324 characters?

If a separator-defined piece is already 324 characters long and there is no suitable separator inside it, the splitter can preserve that piece instead of arbitrarily cutting it at exactly 50 characters.

Therefore:

```text
chunk_size=50
      ≠
every chunk has exactly / at most 50 characters
```


In [17]:
# CharacterTextSplitter demonstration

text = text_docs[0].page_content

character_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=50,
    chunk_overlap=0
)

character_chunks = character_splitter.split_text(text)

print("Number of chunks:", len(character_chunks))

for i, chunk in enumerate(character_chunks, start=1):
    print(f"\n--- CHUNK {i} ---")
    print("Length:", len(chunk))
    print(chunk)


Number of chunks: 19

--- CHUNK 1 ---
Length: 49
ACME Technologies â€” Employee Policy

1. Working

--- CHUNK 2 ---
Length: 50
Hours

Employees are expected to work from 9:00 AM

--- CHUNK 3 ---
Length: 44
to 6:00 PM,
Monday through Friday. Employees

--- CHUNK 4 ---
Length: 50
should be available during
core working hours from

--- CHUNK 5 ---
Length: 50
10:00 AM to 4:00 PM.

2. Work From Home

Employees

--- CHUNK 6 ---
Length: 42
can work remotely for up to three days per

--- CHUNK 7 ---
Length: 41
week.
Remote work must be approved by the

--- CHUNK 8 ---
Length: 47
employee's manager.

3. Leave Policy

Employees

--- CHUNK 9 ---
Length: 42
receive 24 days of paid leave per calendar

--- CHUNK 10 ---
Length: 49
year.
Leave requests should normally be submitted

--- CHUNK 11 ---
Length: 43
at least three
working days in advance.

4.

--- CHUNK 12 ---
Length: 35
Security Policy

Employees must use

--- CHUNK 13 ---
Length: 28
company-managed devices when

--- CHUNK 14 ---
Length: 43

# 1️⃣1️⃣ RecursiveCharacterTextSplitter ⭐

For general-purpose RAG, this is one of the most important splitters to understand.

Typical separators:

Paragraphs, Lines , Words , Characters

```python
[
    "\n\n",
    "\n",
    " ",
    ""
]
```

It tries to preserve larger meaningful boundaries first and falls back to smaller boundaries when necessary.


## Recursive Split Flow Diagram

Imagine the text is too large.

```text
                    FULL TEXT
                        │
                        ▼
                Try separator
                    "\n\n"
                        │
                 Too large?
                  ┌─────┴─────┐
                 NO           YES
                 │             │
                 ▼             ▼
             keep split     Try "\n"
                               │
                          Too large?
                         ┌─────┴─────┐
                        NO           YES
                        │             │
                        ▼             ▼
                    keep split      Try " "
                                      │
                                 Too large?
                                ┌─────┴─────┐
                               NO           YES
                               │             │
                               ▼             ▼
                           keep split       Try ""
                                             │
                                             ▼
                                      character-level
                                        fallback
```

### The key idea

**Recursive does not mean "split randomly again and again."**

It means:

> Try a preferred separator first. If the resulting pieces are still too large, recursively use a smaller separator.


In [18]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=0
)

recursive_chunks = recursive_splitter.split_text(text)

print("Number of chunks:", len(recursive_chunks))

for i, chunk in enumerate(recursive_chunks, start=1):
    print(f"\n--- CHUNK {i} ---")
    print("Length:", len(chunk))
    print(chunk)


Number of chunks: 30

--- CHUNK 1 ---
Length: 37
ACME Technologies â€” Employee Policy

--- CHUNK 2 ---
Length: 16
1. Working Hours

--- CHUNK 3 ---
Length: 46
Employees are expected to work from 9:00 AM to

--- CHUNK 4 ---
Length: 8
6:00 PM,

--- CHUNK 5 ---
Length: 42
Monday through Friday. Employees should be

--- CHUNK 6 ---
Length: 16
available during

--- CHUNK 7 ---
Length: 44
core working hours from 10:00 AM to 4:00 PM.

--- CHUNK 8 ---
Length: 17
2. Work From Home

--- CHUNK 9 ---
Length: 48
Employees can work remotely for up to three days

--- CHUNK 10 ---
Length: 9
per week.

--- CHUNK 11 ---
Length: 46
Remote work must be approved by the employee's

--- CHUNK 12 ---
Length: 8
manager.

--- CHUNK 13 ---
Length: 15
3. Leave Policy

--- CHUNK 14 ---
Length: 43
Employees receive 24 days of paid leave per

--- CHUNK 15 ---
Length: 14
calendar year.

--- CHUNK 16 ---
Length: 46
Leave requests should normally be submitted at

--- CHUNK 17 ---
Length: 11
least three

--- CHUNK 18 -

# 1️⃣2️⃣ Recursive Splitter — Why Four Separators?

Let's understand each separator.

### Level 1 — `\n\n`

```text
Paragraph boundary
```

Best when you want to preserve complete paragraphs.

### Level 2 — `\n`

```text
Line boundary
```

If a paragraph is too large, try individual lines.

### Level 3 — `" "`

```text
Word boundary
```

If lines are still too large, split around spaces.

### Level 4 — `""`

```text
Character boundary
```

This is the final fallback when even word-level pieces are too large.

So:

```text
\n\n
 ↓
\n
 ↓
" "
 ↓
""
```

means:

```text
larger semantic boundary
        ↓
smaller boundary
        ↓
word boundary
        ↓
character fallback
```


# 1️⃣3️⃣ TokenTextSplitter

LLMs do not fundamentally process text as "characters."

They process **tokens**.

A token can represent:

- part of a word
- a complete word
- punctuation
- spaces or common character sequences

The exact tokenization depends on the tokenizer/model family.

### Why token-based splitting?

If you are worried about a model's context window, token-based sizing can be useful.

Conceptually:

```text
TEXT
 │
 ▼
TOKENIZER
 │
 ├── token 1
 ├── token 2
 ├── token 3
 ├── ...
 └── token N
 │
 ▼
TokenTextSplitter
 │
 ▼
Token-sized chunks
```

### Important

Token count is **not equal to character count**.

For example:

```text
1000 characters
```

does not necessarily mean:

```text
1000 tokens
```

The relationship varies with language, formatting, vocabulary, and tokenizer.


In [19]:
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=0
)

token_chunks = token_splitter.split_text(text)

print("Number of token chunks:", len(token_chunks))

for i, chunk in enumerate(token_chunks[:5], start=1):
    print(f"\n--- CHUNK {i} ---")
    print("Characters in returned text:", len(chunk))
    print(chunk)


Number of token chunks: 5

--- CHUNK 1 ---
Characters in returned text: 196
ACME Technologies â€” Employee Policy

1. Working Hours

Employees are expected to work from 9:00 AM to 6:00 PM,
Monday through Friday. Employees should be available during
core working hours from

--- CHUNK 2 ---
Characters in returned text: 173
 10:00 AM to 4:00 PM.

2. Work From Home

Employees can work remotely for up to three days per week.
Remote work must be approved by the employee's manager.

3. Leave Policy

--- CHUNK 3 ---
Characters in returned text: 228


Employees receive 24 days of paid leave per calendar year.
Leave requests should normally be submitted at least three
working days in advance.

4. Security Policy

Employees must use company-managed devices when accessing
conf

--- CHUNK 4 ---
Characters in returned text: 229
idential company information. Passwords must not be
shared with other employees.

5. Learning and Development

Employees receive an annual learning budget of INR 25,000.
The 

# 1️⃣4️⃣ Token Count vs Character Count

A common interview question is:

> "Can I use character length as a replacement for token length?"

Not reliably.

Consider:

```text
Character count
       ↓
len(text)
```

This counts Python string characters.

Token count requires a tokenizer.

```text
Text
 ↓
Tokenizer
 ↓
Tokens
 ↓
Token count
```

### Example with `tiktoken`

`tiktoken` is commonly used for OpenAI tokenization, but **do not assume its count is the exact count for every other LLM**.

For other model families, use their documented tokenizer where available.


In [20]:
# Optional: OpenAI-style token counting with tiktoken
# Install first if required:
# pip install tiktoken

try:
    import tiktoken

    encoding = tiktoken.get_encoding("cl100k_base")

    sample = text[:1000]
    token_ids = encoding.encode(sample)

    print("Characters:", len(sample))
    print("Tokens using cl100k_base:", len(token_ids))

except ImportError:
    print("tiktoken is not installed. Run: pip install tiktoken")


Characters: 851
Tokens using cl100k_base: 186


# 1️⃣5️⃣ Character vs Recursive vs Token — Visual Comparison

## CharacterTextSplitter

```text
TEXT
 │
 ▼
Separator: "\n\n"
 │
 ▼
Paragraph pieces
 │
 ▼
Merge toward target
 │
 ▼
CHUNKS
```

## RecursiveCharacterTextSplitter

```text
TEXT
 │
 ▼
Try "\n\n"
 │
 ├── suitable → use
 │
 └── too large
       ↓
     Try "\n"
       ↓
     too large
       ↓
     Try " "
       ↓
     too large
       ↓
     Try ""
       ↓
     CHUNKS
```

## TokenTextSplitter

```text
TEXT
 │
 ▼
TOKENIZER
 │
 ▼
TOKENS
 │
 ▼
Token-sized groups
 │
 ▼
CHUNKS
```


# 1️⃣6️⃣ Side-by-Side Comparison

| Feature | Character | Recursive Character | Token |
|---|---|---|---|
| Main unit | Characters + separator pieces | Characters + hierarchical separators | Tokens |
| Separator strategy | One main separator | Multiple fallback separators | Token boundaries |
| Semantic boundary preservation | Depends on separator | Better general-purpose behavior | Primarily token budget |
| Hard token budgeting | ❌ | ❌ | ✅ token-oriented |
| Easy to understand | ✅ | ✅ | Moderate |
| Common RAG choice | Sometimes | ⭐ Very common | Useful for token constraints |
| Model tokenizer dependency | No | No | Yes |

### Practical interpretation

```text
Character
→ "I want to split around this specific separator."

Recursive
→ "I want to preserve good text boundaries and fall back intelligently."

Token
→ "I need chunks sized around token limits."
```


# 1️⃣7️⃣ Chunk Overlap

Suppose:

```python
chunk_size=100
chunk_overlap=20
```

Conceptually:

```text
                 100 chars
        ┌─────────────────────────┐
Chunk 1 │AAAAAAAAAAAAAAAAAAAAAAAAA│
        └─────────────────────────┘
                         │
                         │ 20 chars repeated
                         ▼
        ┌─────────────────────────┐
Chunk 2 │AAAAAAAA BBBBBBBBBBBBBBBB│
        └─────────────────────────┘
          ↑
       overlap
```

The exact behavior depends on the splitter and boundaries.

### Why overlap?

Imagine a sentence is split at a boundary:

```text
Chunk 1: "The employee must submit the form"
Chunk 2: "before Friday to remain compliant."
```

Without overlap, related context can be separated.

Overlap gives the next chunk some surrounding context.

Typical RAG values might be something like:

```python
chunk_size=500
chunk_overlap=50
```

or:

```python
chunk_size=1000
chunk_overlap=100
```

But these are **starting points, not universal rules**.


# 1️⃣8️⃣ End-to-End RAG Preparation Example

Here is the practical pipeline you can use in a RAG project.

```python
loader = TextLoader("data/sample_data.txt")

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
```

Notice the difference:

### `split_text()`

```python
chunks = splitter.split_text(text)
```

Input:

```text
string
```

Output:

```text
list[str]
```

### `split_documents()`

```python
chunks = splitter.split_documents(documents)
```

Input:

```text
list[Document]
```

Output:

```text
list[Document]
```

Using `split_documents()` can preserve document metadata with the resulting chunks, which is very useful for RAG citations and source tracking.


In [21]:
# Recommended RAG-style example

loader = TextLoader("data/sample_data.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

document_chunks = splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Final chunks:", len(document_chunks))

for i, chunk in enumerate(document_chunks[:5], start=1):
    print(f"\n--- CHUNK {i} ---")
    print("Characters:", len(chunk.page_content))
    print("Metadata:", chunk.metadata)
    print(chunk.page_content[:300])


Original documents: 1
Final chunks: 2

--- CHUNK 1 ---
Characters: 369
Metadata: {'source': 'data/sample_data.txt'}
ACME Technologies â€” Employee Policy

1. Working Hours

Employees are expected to work from 9:00 AM to 6:00 PM,
Monday through Friday. Employees should be available during
core working hours from 10:00 AM to 4:00 PM.

2. Work From Home

Employees can work remotely for up to three days per week.
Rem

--- CHUNK 2 ---
Characters: 497
Metadata: {'source': 'data/sample_data.txt'}
3. Leave Policy

Employees receive 24 days of paid leave per calendar year.
Leave requests should normally be submitted at least three
working days in advance.

4. Security Policy

Employees must use company-managed devices when accessing
confidential company information. Passwords must not be
share


# 1️⃣9️⃣ Common Mistakes to Avoid

### ❌ Mistake 1

> "`chunk_size=50` means every CharacterTextSplitter chunk must have 50 characters."

Not necessarily.

### ❌ Mistake 2

> "PDF page count equals RAG chunk count."

No.

```text
PDF pages
   ↓
Loader documents
   ↓
Splitter chunks
```

One page can become many chunks.

### ❌ Mistake 3

> "Every LLM uses the same tokenization."

No.

Different model families can use different tokenizers.

### ❌ Mistake 4

> "More overlap is always better."

No. Excessive overlap can increase the number of chunks, storage, retrieval candidates, and processing cost.

### ❌ Mistake 5

> "Bigger chunks always provide better context."

Not necessarily. Larger chunks can also introduce irrelevant content and reduce retrieval precision.



## Final Architecture

```text
┌──────────────────────────────────────────────────────┐
│                    SOURCE DATA                       │
│   TXT        PDF        DOCX        WEB              │
└────┬─────────┬──────────┬───────────┬────────────────┘
     │         │          │           │
     └─────────┴──────────┴───────────┘
                       │
                       ▼
              ┌─────────────────┐
              │ Document Loader  │
              └────────┬────────┘
                       │
                       ▼
                LangChain Docs
                       │
                       ▼
              ┌─────────────────┐
              │  Text Splitter  │
              └────────┬────────┘
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
      Character    Recursive      Token
          │            │            │
          └────────────┼────────────┘
                       ▼
                    CHUNKS
                       │
                       ▼
                  EMBEDDINGS
                       │
                       ▼
                 VECTOR STORE
                       │
                       ▼
                   RETRIEVER
                       │
                       ▼
                      LLM
                       │
                       ▼
                   RAG ANSWER
```

---

## 🎯 Key takeaway

**Loader = get the data**

**Splitter = divide the data**

**Character = separator-based**

**Recursive = hierarchical separator fallback**

**Token = token-budget-oriented**

**Chunk size = a target/constraint interpreted by the splitter, not a universal promise that every output chunk will have exactly that many characters.**
